# 🤝 HACRL: Heterogeneous Agent Collaborative RL — Local Training Demo

**Paper**: [arXiv 2603.02604](https://arxiv.org/abs/2603.02604) — *Heterogeneous Agent Collaborative Reinforcement Learning*

**Core idea**: Instead of each LLM doing RL in isolation, heterogeneous models **share verified rollouts** during training.  
A 1.5B and a 7B model both attempt math problems → they share each other's correct solutions → both improve faster.

**Why Strix Halo?** AMD Ryzen AI Max+ has up to 128 GB unified memory at 256 GB/s — enough to hold two LLMs simultaneously and run RL training on them.

> 🦃 *Created by Crunch for [Copilotclaw/trainingdemo](https://github.com/Copilotclaw/trainingdemo)*

## 📖 How HACRL Works

Standard on-policy RL (GRPO/PPO) trains each agent in isolation:
```
Agent A: generate rollouts → reward → advantages → update A
Agent B: generate rollouts → reward → advantages → update B
```

**HACRL (this paper)** — agents share verified rollouts:
```
Agent A generates rollouts ──┐
                             ├──► shared verified pool
Agent B generates rollouts ──┘         │
                             ┌─────────┴──────────┐
                             ▼                    ▼
                A trains on A+B rollouts   B trains on B+A rollouts
                (importance-weighted)      (importance-weighted)
```

**HACPO mechanisms** (four from the paper):
1. **Dual-clip importance weighting** — caps IS ratio to handle capability gaps
2. **Distribution shift monitoring** — stops sharing if KL divergence is too high
3. **Adaptive rollout selection** — only share high-reward (verified correct) rollouts
4. **Asymmetric λ scaling** — weaker agent gets proportionally more from stronger

**Result**: +3.3% vs GSPO baseline, using **half the rollout cost**.

## 💻 Strix Halo Hardware Context

| Spec | Value | Why it matters |
|------|-------|----------------|
| Unified RAM | up to 128 GB LPDDR5X-8000 | Holds 2× models simultaneously |
| GPU | Radeon 8060S (40 CU, RDNA 3.5) | ROCm RL training |
| Bandwidth | 256 GB/s | Fast weight access during backprop |
| NPU | 50 TOPS XDNA 2 | Lightweight reward inference |

**Memory budget for this demo (LoRA training):**
| Component | Size |
|-----------|------|
| Qwen2.5-Math-1.5B bf16 weights | ~3 GB |
| Qwen2.5-Math-7B bf16 weights | ~14 GB |
| LoRA adapters + optimizer states | ~8 GB |
| Rollout buffers + activations | ~10 GB |
| **Total** | **~35 GB** — very comfortable on 128 GB |

**ROCm environment setup** (run in terminal before launching notebook):
```bash
export HSA_OVERRIDE_GFX_VERSION=11.0.0   # Strix Halo = gfx1150
export ROCR_VISIBLE_DEVICES=0
export PYTORCH_HIP_ALLOC_CONF=expandable_segments:True
```

## 🛠️ Install Requirements

```bash
# ROCm PyTorch for Strix Halo
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.2

# RL + fine-tuning libraries
pip install trl transformers accelerate datasets peft sympy matplotlib
```

In [ ]:
import torch
import re
import math
import random
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Optional

# ── Detect hardware ────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} ({vram:.1f} GB)")
    DEVICE = "cuda"
else:
    print("CPU mode (slower but still runnable)")
    DEVICE = "cpu"

print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

## 🤖 Load Two Heterogeneous Agents

We use math-specialized Qwen2.5 models at two sizes.  
Only **LoRA adapters** are trained — base weights stay frozen, keeping memory usage low.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

MODEL_A = "Qwen/Qwen2.5-Math-1.5B"   # smaller / faster rollout generation
MODEL_B = "Qwen/Qwen2.5-Math-7B"     # larger / stronger reasoning

LORA_CFG = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

def load_agent(model_name: str, label: str):
    print(f"Loading {label} ({model_name})...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model = get_peft_model(model, LORA_CFG)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  {trainable/1e3:.0f}K trainable / {total/1e6:.0f}M total ({100*trainable/total:.2f}%)")
    return model, tokenizer

agent_a, tok_a = load_agent(MODEL_A, "Agent A (1.5B)")
agent_b, tok_b = load_agent(MODEL_B, "Agent B (7B)")

## 📐 Task: Math Reasoning with Verifiable Rewards

Models solve arithmetic and linear algebra problems.  
Reward is **outcome-based**: +1 if final numerical answer correct, 0 otherwise.  
No judge model needed — answer verification is exact.

Models must produce `<answer>N</answer>` tags for reliable extraction.

In [ ]:
def generate_problems(n: int = 64, seed: int = 42) -> List[Dict]:
    rng = random.Random(seed)
    problems = []
    for _ in range(n):
        kind = rng.choice(["add", "mul", "linear"])
        if kind == "add":
            a, b = rng.randint(1, 500), rng.randint(1, 500)
            q, ans = f"What is {a} + {b}?", a + b
        elif kind == "mul":
            a, b = rng.randint(2, 50), rng.randint(2, 50)
            q, ans = f"What is {a} x {b}?", a * b
        else:
            a = rng.randint(1, 20)
            ans = rng.randint(1, 20)
            b = rng.randint(0, 50)
            c = a * ans + b
            q = f"Solve for x: {a}x + {b} = {c}"
        problems.append({"question": q, "answer": float(ans)})
    return problems

PROMPT = """Solve the following math problem step by step.
Put your final numerical answer inside <answer></answer> tags.

Problem: {question}

Solution:"""

def extract_answer(text: str) -> Optional[float]:
    m = re.search(r"<answer>\s*([\-\d\.]+)\s*</answer>", text)
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            pass
    nums = re.findall(r"\-?\d+\.?\d*", text)
    return float(nums[-1]) if nums else None

def reward(generated: str, correct: float) -> float:
    pred = extract_answer(generated)
    return 1.0 if pred is not None and abs(pred - correct) < 0.5 else 0.0

problems = generate_problems(64)
print(f"{len(problems)} problems generated")
for p in problems[:3]:
    print(f"  {p['question']} -> {p['answer']}")

## 🎲 Rollout Generation

Each agent generates multiple attempts per problem (group sampling, same as GRPO).  
We record **log probabilities** of every output token — needed for importance weighting later.

In [ ]:
@dataclass
class Rollout:
    question: str
    correct_answer: float
    prompt_ids: torch.Tensor    # [prompt_len]
    output_ids: torch.Tensor    # [output_len]
    log_probs: torch.Tensor     # [output_len] log probs under generating policy
    rew: float
    agent_id: str
    text: str

@torch.no_grad()
def generate_rollouts(
    model, tokenizer, problems: List[Dict], agent_id: str,
    n_per: int = 4, max_new: int = 256, temp: float = 0.8
) -> List[Rollout]:
    model.eval()
    rollouts = []
    for prob in problems:
        prompt = PROMPT.format(question=prob["question"])
        inp = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        input_ids = inp["input_ids"].to(DEVICE)
        prompt_len = input_ids.shape[1]
        for _ in range(n_per):
            out = model.generate(
                input_ids, max_new_tokens=max_new,
                do_sample=(temp > 0), temperature=max(temp, 1e-6),
                pad_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=True, output_scores=True,
            )
            output_ids = out.sequences[0, prompt_len:]
            # Log prob of each generated token
            if out.scores:
                lp = torch.stack([
                    torch.nn.functional.log_softmax(s[0], dim=-1)[output_ids[i]]
                    for i, s in enumerate(out.scores) if i < len(output_ids)
                ])
            else:
                lp = torch.zeros(len(output_ids))
            text = tokenizer.decode(output_ids, skip_special_tokens=True)
            rollouts.append(Rollout(
                question=prob["question"], correct_answer=prob["answer"],
                prompt_ids=input_ids[0].cpu(), output_ids=output_ids.cpu(),
                log_probs=lp.cpu(), rew=reward(text, prob["answer"]),
                agent_id=agent_id, text=text,
            ))
    return rollouts

print("Rollout generator ready")

## ⚙️ HACPO Core: Importance Weighting + Distribution Guards

**GRPO advantage**: normalize rewards within each problem's rollout group
$$A_i = \frac{r_i - \bar{r}}{\sigma(r) + \epsilon}$$

**Cross-agent importance weight**: correct for off-policy distribution
$$w_{A \leftarrow B} = \exp\!\left(\sum_t \log \pi_A(a_t|s_t) - \log \pi_B(a_t|s_t)\right) \quad \text{clipped to } [c_l, c_h]$$

**Asymmetric lambda**: weaker agent gets more from stronger agent
$$\lambda_{A \leftarrow B} = \sigma\!\left(\frac{\text{acc}_B - \text{acc}_A}{T}\right)$$

In [ ]:
from collections import defaultdict

def grpo_advantages(rollouts: List[Rollout]) -> torch.Tensor:
    groups = defaultdict(list)
    for i, r in enumerate(rollouts):
        groups[r.question].append(i)
    adv = torch.zeros(len(rollouts))
    for idxs in groups.values():
        rews = torch.tensor([rollouts[i].rew for i in idxs])
        a = (rews - rews.mean()) / (rews.std() + 1e-8) if rews.std() > 1e-6 else rews - rews.mean()
        for j, i in enumerate(idxs):
            adv[i] = a[j]
    return adv

@torch.no_grad()
def log_probs_under(model, r: Rollout) -> torch.Tensor:
    prompt = r.prompt_ids.unsqueeze(0).to(DEVICE)
    output = r.output_ids.to(DEVICE)
    full = torch.cat([prompt, output.unsqueeze(0)], dim=1)
    logits = model(full).logits
    plen = prompt.shape[1]
    lp = torch.nn.functional.log_softmax(logits[0, plen-1:-1], dim=-1)
    return lp[torch.arange(len(output)), output].cpu()

def importance_weight(current_lp: torch.Tensor, ref_lp: torch.Tensor,
                      lo: float = 0.1, hi: float = 5.0) -> float:
    log_r = (current_lp.sum() - ref_lp.sum()).item()
    return max(lo, min(hi, math.exp(max(-10, min(10, log_r)))))

def kl_approx(lp_a: torch.Tensor, lp_b: torch.Tensor) -> float:
    return (lp_a - lp_b).mean().abs().item()

def asym_lambda(acc_source: float, acc_target: float, T: float = 0.1) -> float:
    return 1.0 / (1.0 + math.exp(-(acc_source - acc_target) / T))

print("HACPO utilities ready:")
print("  grpo_advantages   -- group reward normalisation")
print("  importance_weight -- dual-clipped IS ratio")
print("  kl_approx         -- distribution shift monitor")
print("  asym_lambda       -- asymmetric scaling")

## 🔄 HACPO Training Loop

Each iteration:
1. Both agents generate rollouts on the same mini-batch
2. Compute GRPO advantages within each agent's rollout group
3. Measure KL distance between policies
4. **Own-policy update**: each agent trains on its own rollouts (standard GRPO)
5. **Cross-agent update** *(if KL < threshold)*: A also trains on B's correct rollouts, importance-weighted and scaled by asymmetric λ
6. Gradient step for both

In [ ]:
from torch.optim import AdamW

@dataclass
class HACPOConfig:
    n_iterations: int = 5        # increase to 20-50 for real training
    n_rollouts: int = 4          # rollouts per problem (G in GRPO)
    n_problems: int = 8          # problems per mini-batch
    lr: float = 1e-5
    kl_threshold: float = 0.5    # max KL before stopping cross-agent sharing
    clip_eps: float = 0.2        # PPO clip
    is_lo: float = 0.1           # importance weight lower clip
    is_hi: float = 5.0           # importance weight upper clip
    cross_w: float = 0.3         # base cross-agent loss weight

def hacpo_loss(model, r: Rollout, advantage: float,
               ref_lp: Optional[torch.Tensor] = None,
               is_w: float = 1.0, clip_eps: float = 0.2) -> torch.Tensor:
    prompt = r.prompt_ids.unsqueeze(0).to(DEVICE)
    output = r.output_ids.to(DEVICE)
    full = torch.cat([prompt, output.unsqueeze(0)], dim=1)
    logits = model(full).logits
    plen = prompt.shape[1]
    lp = torch.nn.functional.log_softmax(logits[0, plen-1:-1], dim=-1)
    curr_lp = lp[torch.arange(len(output)), output]
    base_lp = (ref_lp if ref_lp is not None else r.log_probs).to(DEVICE)
    ratio = torch.exp((curr_lp - base_lp).sum().clamp(-5, 5))
    clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)
    return -(is_w * torch.min(ratio * advantage, clipped * advantage))

def run_hacpo(agent_a, tok_a, agent_b, tok_b, problems, cfg: HACPOConfig):
    opt_a = AdamW([p for p in agent_a.parameters() if p.requires_grad], lr=cfg.lr)
    opt_b = AdamW([p for p in agent_b.parameters() if p.requires_grad], lr=cfg.lr)
    history = {"iter": [], "acc_a": [], "acc_b": [], "sharing": []}

    for it in range(cfg.n_iterations):
        batch = random.sample(problems, min(cfg.n_problems, len(problems)))

        # Step 1: Generate rollouts
        print(f"\n[Iter {it+1}/{cfg.n_iterations}] Generating rollouts...")
        ra = generate_rollouts(agent_a, tok_a, batch, "A", n_per=cfg.n_rollouts)
        rb = generate_rollouts(agent_b, tok_b, batch, "B", n_per=cfg.n_rollouts)
        acc_a = sum(r.rew for r in ra) / len(ra)
        acc_b = sum(r.rew for r in rb) / len(rb)
        print(f"  Accuracy: A={acc_a:.2%}  B={acc_b:.2%}")

        # Step 2: GRPO advantages
        adv_a, adv_b = grpo_advantages(ra), grpo_advantages(rb)

        # Step 3: KL distance (sample a few B rollouts)
        sample_b = [r for r in rb if r.rew > 0][:4]
        kls = [kl_approx(log_probs_under(agent_a, r), r.log_probs) for r in sample_b]
        avg_kl = float(np.mean(kls)) if kls else float("inf")
        sharing = avg_kl < cfg.kl_threshold and bool(sample_b)
        lam = asym_lambda(acc_b, acc_a)
        print(f"  KL={avg_kl:.3f} (thresh {cfg.kl_threshold})  lambda={lam:.2f}  sharing={'ON' if sharing else 'OFF'}")

        # Step 4: Update Agent A
        agent_a.train()
        opt_a.zero_grad()
        loss_a = sum(
            hacpo_loss(agent_a, r, adv_a[i].item(), clip_eps=cfg.clip_eps)
            for i, r in enumerate(ra) if adv_a[i].abs() > 1e-8
        )
        if sharing:
            for r in sample_b:
                lp_a = log_probs_under(agent_a, r)
                w = importance_weight(lp_a, r.log_probs, cfg.is_lo, cfg.is_hi)
                loss_a = loss_a + hacpo_loss(
                    agent_a, r, 1.0, ref_lp=r.log_probs,
                    is_w=w * cfg.cross_w * lam, clip_eps=cfg.clip_eps
                )
        (loss_a / max(len(ra), 1)).backward()
        torch.nn.utils.clip_grad_norm_(agent_a.parameters(), 1.0)
        opt_a.step()

        # Step 5: Update Agent B (own policy only for simplicity)
        agent_b.train()
        opt_b.zero_grad()
        loss_b = sum(
            hacpo_loss(agent_b, r, adv_b[i].item(), clip_eps=cfg.clip_eps)
            for i, r in enumerate(rb) if adv_b[i].abs() > 1e-8
        )
        (loss_b / max(len(rb), 1)).backward()
        torch.nn.utils.clip_grad_norm_(agent_b.parameters(), 1.0)
        opt_b.step()

        loss_a_v = (loss_a / max(len(ra), 1)).item()
        loss_b_v = (loss_b / max(len(rb), 1)).item()
        print(f"  Loss: A={loss_a_v:.4f}  B={loss_b_v:.4f}")
        history["iter"].append(it + 1)
        history["acc_a"].append(acc_a)
        history["acc_b"].append(acc_b)
        history["sharing"].append(sharing)

    return history

print("run_hacpo() ready")

## 🏃 Run Training

Start small to verify end-to-end. Increase `n_iterations` to 30+ and `n_problems` to 32 for a real run.

In [ ]:
cfg = HACPOConfig(
    n_iterations=3,    # ← 30-50 for real training
    n_rollouts=4,
    n_problems=4,      # ← 32 for real training
    lr=1e-5,
    kl_threshold=0.5,
    cross_w=0.3,
)

history = run_hacpo(agent_a, tok_a, agent_b, tok_b, problems, cfg)

## 📊 Results

In [ ]:
# Evaluate on held-out problems
eval_probs = generate_problems(32, seed=999)

@torch.no_grad()
def evaluate(model, tok, probs, aid):
    rs = generate_rollouts(model, tok, probs, aid, n_per=1, temp=0.0)
    return sum(r.rew for r in rs) / len(rs)

print("Evaluating...")
final_a = evaluate(agent_a, tok_a, eval_probs, "A")
final_b = evaluate(agent_b, tok_b, eval_probs, "B")
print(f"\nFinal eval accuracy:")
print(f"  Agent A (1.5B): {final_a:.2%}")
print(f"  Agent B (7B):   {final_b:.2%}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history["iter"], history["acc_a"], "b-o", label="Agent A (1.5B)", lw=2)
    ax1.plot(history["iter"], history["acc_b"], "r-s", label="Agent B (7B)", lw=2)
    ax1.set_xlabel("Iteration"); ax1.set_ylabel("Rollout Accuracy")
    ax1.set_title("HACPO: Accuracy per Iteration"); ax1.legend(); ax1.grid(alpha=0.3)

    colors = ["green" if x else "salmon" for x in history["sharing"]]
    ax2.bar(history["iter"], [1 if x else 0.3 for x in history["sharing"]], color=colors, alpha=0.8)
    ax2.set_xlabel("Iteration"); ax2.set_title("Cross-Agent Sharing: Active vs Skipped")
    ax2.set_yticks([0, 1]); ax2.set_yticklabels(["Skipped", "Active"]); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("hacrl-training-curves.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved hacrl-training-curves.png")
except ImportError:
    print("matplotlib not installed — skipping plot")
    print("History:", history)

## 💾 Save LoRA Adapters

In [ ]:
import os
os.makedirs("adapters", exist_ok=True)

agent_a.save_pretrained("adapters/agent_a_hacpo")
tok_a.save_pretrained("adapters/agent_a_hacpo")
agent_b.save_pretrained("adapters/agent_b_hacpo")
tok_b.save_pretrained("adapters/agent_b_hacpo")

for path in ["adapters/agent_a_hacpo", "adapters/agent_b_hacpo"]:
    total = sum(os.path.getsize(os.path.join(dp, f))
                for dp, _, fnames in os.walk(path) for f in fnames)
    print(f"{path}: {total/1e6:.1f} MB")

## 🔧 Strix Halo Tips

### Larger model pairs (all fit in 128 GB)
| Agent A | Agent B | VRAM estimate |
|---------|---------|---------------|
| Qwen2.5-0.5B | Qwen2.5-1.5B | ~4 GB |
| Qwen2.5-1.5B | Qwen2.5-7B | ~20 GB |
| Qwen2.5-7B | Qwen2.5-14B | ~45 GB |
| Qwen2.5-7B | Qwen2.5-32B | ~65 GB |
| Llama-3.1-8B | Llama-3.3-70B (4-bit) | ~55 GB |

### ROCm env vars (before launching)
```bash
export HSA_OVERRIDE_GFX_VERSION=11.0.0   # Strix Halo gfx1150
export PYTORCH_HIP_ALLOC_CONF=expandable_segments:True
export ROCR_VISIBLE_DEVICES=0
```

### Speed up with vLLM batched rollouts
```python
# pip install vllm  (with ROCm build)
from vllm import LLM, SamplingParams
llm = LLM(model=MODEL_A, dtype="bfloat16", gpu_memory_utilization=0.4)
# Much faster than HF generate() for rollout generation phase
```

### Further reading
- [HACRL paper](https://arxiv.org/abs/2603.02604) — the source
- [GRPO paper](https://arxiv.org/abs/2402.03300) — base RL algorithm
- [TRL GRPOTrainer](https://huggingface.co/docs/trl/grpo_trainer) — production-grade base to extend
- [Strix Halo ROCm setup](https://github.com/Gygeek/Framework-strix-halo-llm-setup)
- [Level1Techs benchmarks](https://forum.level1techs.com/t/strix-halo-ryzen-ai-max-395-llm-benchmark-results/233796)